# Experiment 1: building the tables one at a time

This notebook is a **thin shell over `ex1_results.py`**, not a copy of it. Every
metric, every file path and every check comes from that module by import, so a
fix in one place fixes both and the two can never disagree.

Put this notebook **next to `ex1_results.py`**, in `fourarm/analysis/ex1/`.

## How to use it

- Run **Setup** once. Everything after it is independent.
- Each question has two cells: a **build** cell that calls the script's own
  function, and a **scratch** cell holding the raw ingredients so you can try a
  different shape without touching the script.
- Nothing writes to disk until the last section. `write=False` throughout.

## One open decision, in Q1

`tab:ex1:constraints` may not earn its place. Q1 builds it and an alternative
side by side so the comparison is on the page rather than in the abstract.

## Setup

In [1]:
import os, sys, json, collections, importlib, pathlib

# Find ex1_results.py: beside the notebook, or anywhere above it.
here = pathlib.Path.cwd()
for cand in [here] + list(here.parents):
    if (cand / "ex1_results.py").exists():
        sys.path.insert(0, str(cand))
        break
else:
    raise SystemExit("ex1_results.py not found. Put this notebook beside it.")

import ex1_results as R
importlib.reload(R)          # re-run this cell after editing the script

ROOT = R.find_root()
FILES, IMAGE, SCHEMES, MISSING = R.resolve_files(ROOT)
if MISSING:
    R.report_missing(ROOT, MISSING)
REF = R.reference_lines(ROOT)

print("root      %s" % ROOT)
print("run files %d (%s)" % (len(FILES), dict(SCHEMES)))
print("image-on  %s" % IMAGE)
print("cast A    width-blind %.1f, chance %.1f"
      % (REF["casta"]["width_blind"], REF["casta"]["chance"]))
print("cast B    width-blind %.1f, chance %.1f"
      % (REF["castb"]["width_blind"], REF["castb"]["chance"]))

root      /Users/erinsarlak/Downloads/MastersDissertation/fourarm
run files 30 ({'new': 30})
image-on  out/ex1_casta_gpt_nowidth-anon_r3_image.jsonl
cast A    width-blind 74.9, chance 30.5
cast B    width-blind 69.2, chance 34.6


In [2]:
# The integrity gate. Nothing below is trustworthy if this does not pass.
R.integrity_gate(ROOT, FILES, verbose=True)


Integrity gate
--------  ------  ------------------  --------------------------------------  ----  -------  --------  ------  ----
cast      model   condition           file                                    rows  repeats  balanced  errors  rung
--------  ------  ------------------  --------------------------------------  ----  -------  --------  ------  ----
casta     gemini  Anonymous           ex1_casta_gemini_anon_r3.jsonl          486   3        yes       0       ok  
casta     gemini  Full Information    ex1_casta_gemini_full_r3.jsonl          486   3        yes       0       ok  
casta     gemini  Legal-Arm Control   ex1_casta_gemini_givenset_r1.jsonl      162   1        yes       0       ok  
casta     gemini  No Rules            ex1_casta_gemini_norules_r3.jsonl       486   3        yes       0       ok  
casta     gemini  No Width            ex1_casta_gemini_nowidth_r3.jsonl       486   3        yes       0       ok  
casta     gemini  No Width + Anon.    ex1_casta_gemini_n

In [3]:
# Convenience. show() renders a table as a DataFrame when pandas is available,
# and falls back to the script's plain printer when it is not.
try:
    import pandas as pd
    pd.set_option("display.max_colwidth", 60)
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False

def show(headers, rows, title=""):
    if HAVE_PANDAS:
        if title:
            print(title)
        return pd.DataFrame(rows, columns=headers)
    R.print_table(title, headers, rows)

def rows_for(model, cond, cast="casta"):
    """The raw run rows for one cell."""
    return R.load(ROOT, FILES[(cast, model, cond)])

MODELS = R.MODELS
print("helpers ready. models:", MODELS)
print("conditions:", list(R.CONDITIONS))

helpers ready. models: ['gemini', 'gpt', 'qwen']
conditions: ['full', 'anon', 'swap', 'nowidth', 'nowidth-anon', 'nowidth-swap', 'norules', 'givenset']


---

# Q1. Can the model apply a stated constraint, and does the kind of comparison matter?

Two things have to come out of this question:

1. **The three-way split between models.** Nothing later in the chapter makes
   sense without it.
2. **Whether the type of comparison matters.** A defensive null: if the numeric
   comparison were harder than the list lookup, the width effect could be
   re-read as "these models are bad at arithmetic."

The current table answers 2 and buries 1.

### 1a. The current table, `tab:ex1:constraints`

In [4]:
R.table_constraints(ROOT, FILES, show=True, write=False)


tab:ex1:constraints   Legality by constraint, Full Information
---------------  ----  ------------------  -------------  ------------  ------------
Constraint       Rule  Reasoning demanded  Gemini         GPT           Qwen        
---------------  ----  ------------------  -------------  ------------  ------------
Reach            R4    list lookup         100.0 (n=213)  99.0 (n=205)  77.9 (n=213)
Grasp            R3    numeric comparison  100.0 (n=153)  96.7 (n=151)  73.9 (n=153)
Delicacy         R3    boolean check       100.0 (n=9)    100.0 (n=6)   44.4 (n=9)  
All constraints  --    --                  100.0 (n=378)  98.1 (n=365)  75.7 (n=378)
---------------  ----  ------------------  -------------  ------------  ------------
reach minus grasp:
  Gemini  +0.0 [-1.8, +2.4]
  GPT     +2.3 [-0.8, +6.6]
  Qwen    +4.1 [-4.7, +13.1]
Delicacy binds only nine states. Draw no conclusion from that row.


**The case against it.** Twelve cells. Three are the delicacy row, which rests
on n=9 and n=6 and supports nothing. Of the remaining nine, six are at ceiling
between 96.7 and 100. What is left is one comparison, reach against grasp,
which spans zero for every model.

A null with no internal structure does not need a grid. It needs a sentence
carrying the two intervals.

**The case for keeping it.** The defensive job is real, and an examiner asking
"is the grasp constraint just harder?" is answered faster by a table than by
prose.

**The case against the case for.** That reader is answered by one sentence with
the numbers in it, and the table costs a float, a caption and a page position.

### 1b. The alternative: a competence profile

In [5]:
# What Q1 actually needs: where each model's errors fall, and what happens when
# the answer is handed to it. Built here rather than in the script, so you can
# change it freely before deciding.
headers = ["Model", "Picking trials correct", "Zero-legal refused",
           "Grasp-binding legality", "With legal set printed"]
rows, store = [], {}
for m in MODELS:
    full = rows_for(m, "full")
    given = rows_for(m, "givenset")

    kp, np_ = R.legality_any_cause(ROOT, FILES, m)      # any binding cause
    kr, nr = R.refusal_trial(full)
    kg, ng = R.legality(full)                            # grasp-binding subset
    kl, nl = R.legality(given)

    rows.append([R.MODEL_LABEL[m],
                 "%d/%d = %.1f" % (kp, np_, R.pct(kp, np_)),
                 "%d/%d = %.1f" % (kr, nr, R.pct(kr, nr)),
                 R.fmt_ci(kg, ng),
                 R.fmt_ci(kl, nl)])
    store[m] = dict(pick=(kp, np_), refuse=(kr, nr), grasp=(kg, ng),
                    given=(kl, nl))

show(headers, rows, "Competence profile at Full Information")


Competence profile at Full Information
------  ----------------------  ------------------  ----------------------  ----------------------
Model   Picking trials correct  Zero-legal refused  Grasp-binding legality  With legal set printed
------  ----------------------  ------------------  ----------------------  ----------------------
Gemini  378/378 = 100.0         101/108 = 93.5      100.0 [98.7, 100.0]     100.0 [96.2, 100.0]   
GPT     358/365 = 98.1          96/108 = 88.9       97.5 [95.0, 98.8]       100.0 [96.1, 100.0]   
Qwen    286/378 = 75.7          0/108 = 0.0         75.3 [70.1, 80.0]       95.8 [89.8, 98.4]     
------  ----------------------  ------------------  ----------------------  ----------------------


In [6]:
# The lift from printing the legal arm set. This is what licenses saying that
# Qwen's failure is in applying constraints rather than in parsing or format.
print("Legality with the legal set printed, minus Full Information\n")
for m in MODELS:
    d = R.newcombe(*store[m]["given"], *store[m]["grasp"])
    print("  %-7s %+.1f [%+.1f, %+.1f]%s"
          % (R.MODEL_LABEL[m], d[0], d[1], d[2],
             "" if R.spans_zero(d[1], d[2]) else "   EXCLUDES ZERO"))

Legality with the legal set printed, minus Full Information

  Gemini  +0.0 [-3.8, +1.3]
  GPT     +2.5 [-1.7, +5.0]
  Qwen    +20.5 [+12.9, +26.4]   EXCLUDES ZERO


**Read the profile row by row.**

Gemini answers every picking trial correctly. Its only Full Information errors
are failures to refuse. GPT is close behind. Qwen fails on a quarter of picking
trials and never refuses at all.

That is the three-way split, stated in a way that also says *what kind* of
failure each model makes. The current table cannot say that.

### 1c. The constraint-type null, as prose

In [7]:
# If tab:ex1:constraints goes, this is what replaces it. Print it in the exact
# form it would appear in the chapter, so the sentence can be checked against
# the numbers rather than retyped from them.
print("Reach (a list lookup) against grasp (a numeric comparison), "
      "Full Information:\n")
for m in MODELS:
    k_r = n_r = k_g = n_g = 0
    for r in rows_for(m, "full"):
        if r.get("zero_legal") or r.get("result") not in ("valid", "rejected"):
            continue
        if r.get("binding_cause") == "reach":
            n_r += 1; k_r += r["result"] == "valid"
        elif r.get("binding_cause") == "grasp":
            n_g += 1; k_g += r["result"] == "valid"
    d = R.newcombe(k_r, n_r, k_g, n_g)
    print("  %-7s reach %5.1f (n=%d), grasp %5.1f (n=%d), "
          "difference %+.1f [%+.1f, %+.1f]"
          % (R.MODEL_LABEL[m], R.pct(k_r, n_r), n_r, R.pct(k_g, n_g), n_g,
             d[0], d[1], d[2]))
print("\nResolution: the widest of these intervals is the size of arithmetic")
print("penalty the data could NOT have detected. Word the claim accordingly.")

Reach (a list lookup) against grasp (a numeric comparison), Full Information:

  Gemini  reach 100.0 (n=213), grasp 100.0 (n=153), difference +0.0 [-1.8, +2.4]
  GPT     reach  99.0 (n=205), grasp  96.7 (n=151), difference +2.3 [-0.8, +6.6]
  Qwen    reach  77.9 (n=213), grasp  73.9 (n=153), difference +4.1 [-4.7, +13.1]

Resolution: the widest of these intervals is the size of arithmetic
penalty the data could NOT have detected. Word the claim accordingly.


**Decision to make here.** Keep `tab:ex1:constraints`, replace it with the
competence profile, or keep both. My recommendation is **replace**: the profile
answers the more important half of Q1, and the constraint null survives as the
sentence printed above.

If you replace it, the label becomes `tab:ex1:baseline` and the file
`tables/ex1_baseline.tex` follows automatically.

### 1d. Where the errors actually fall

In [8]:
# Supporting detail for the profile. Not necessarily a table, but the numbers
# behind "its only errors are failures to refuse".
for m in MODELS:
    onzero, onpick = collections.Counter(), collections.Counter()
    for r in rows_for(m, "full"):
        vc = r.get("violation_cause")
        if not vc:
            continue
        (onzero if r.get("zero_legal") else onpick)[vc] += 1
    print("%-7s picking states   %s" % (R.MODEL_LABEL[m], dict(onpick) or "none"))
    print("%-7s zero-legal states %s" % ("", dict(onzero) or "none"))

Gemini  picking states   none
        zero-legal states {'no_route': 7}
GPT     picking states   {'no_route': 2, 'grasp': 5}
        zero-legal states {'no_route': 8, 'grasp': 2, 'delicate': 2}
Qwen    picking states   {'arm_state': 26, 'reach': 21, 'grasp': 14, 'delicate': 30, 'no_route': 1}
        zero-legal states {'grasp': 27, 'arm_state': 41, 'reach': 31, 'delicate': 9}


In [9]:
# Legal-Arm Control, split by state type. The pooled "unlisted arm" share mixes
# two different failures and should not be reported as one number.
probes = json.load(open(os.path.join(ROOT, "probes/ex1_v2.json")))["probes"]
IDLE = {}
for p in probes:
    pv = p["provenance"]
    IDLE[(pv["source"], pv["seq"], pv["round"])] = {
        a["name"] for a in p["state"]["arms"]
        if a.get("state") == "IDLE" and not a.get("disabled")}

headers = ["Model", "Picking proposals", "unlisted", "Zero-legal proposals",
           "unlisted"]
rows = []
for m in MODELS:
    c = collections.Counter()
    for r in rows_for(m, "givenset"):
        arm = (r.get("decision") or {}).get("arm")
        if not arm:
            continue
        where = "zero" if r.get("zero_legal") else "pick"
        c[(where, arm in IDLE[R.scene_key(r)])] += 1
    tp = c[("pick", True)] + c[("pick", False)]
    tz = c[("zero", True)] + c[("zero", False)]
    rows.append([R.MODEL_LABEL[m], tp, c[("pick", False)], tz, c[("zero", False)]])
show(headers, rows, "Legal-Arm Control, proposals naming an arm not on the "
                    "printed list")


Legal-Arm Control, proposals naming an arm not on the printed list
------  -----------------  --------  --------------------  --------
Model   Picking proposals  unlisted  Zero-legal proposals  unlisted
------  -----------------  --------  --------------------  --------
Gemini  126                0         0                     0       
GPT     118                0         0                     0       
Qwen    126                5         34                    31      
------  -----------------  --------  --------------------  --------


---

# Q2. What happens when the declared width is withheld?

In [10]:
R.table_design(ROOT, FILES, REF, show=True, write=False)


tab:ex1:design   Legality across the design, cast A
------  ------------------  -------  --------  -------------------  ---  -----  ------------
Model   Condition           Width    Identity  Legality             n    Scene  Neg. control
------  ------------------  -------  --------  -------------------  ---  -----  ------------
Gemini  Full Information    present  true      100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Anonymous           present  withheld  100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Swapped Names       present  swapped   100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  No Width            absent   true      72.2 [66.8, 77.1]    288  71.9   100.0       
Gemini  No Width + Anon.    absent   withheld  75.3 [70.1, 80.0]    288  77.1   100.0       
Gemini  No Width + Swapped  absent   swapped   76.0 [70.7, 80.5]    287  75.0   100.0       
Gemini  No Rules            --       --        100.0 [98.7, 100.0]  288  100.0  100.0       
Gemini  Legal-Arm

[['Gemini',
  'full',
  'Full Information',
  'present',
  'true',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'anon',
  'Anonymous',
  'present',
  'withheld',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'swap',
  'Swapped Names',
  'present',
  'swapped',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth',
  'No Width',
  'absent',
  'true',
  208,
  288,
  '72.22',
  '71.88',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth-anon',
  'No Width + Anon.',
  'absent',
  'withheld',
  217,
  288,
  '75.35',
  '77.08',
  90,
  90,
  '100.00'],
 ['Gemini',
  'nowidth-swap',
  'No Width + Swapped',
  'absent',
  'swapped',
  218,
  287,
  '75.96',
  '75.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'norules',
  'No Rules',
  '--',
  '--',
  288,
  288,
  '100.00',
  '100.00',
  90,
  90,
  '100.00'],
 ['Gemini',
  'givenset',
  'Legal-Arm Control',
  '--',
  '--',
  96,
  96,
  '100.00',
  '10

In [11]:
R.table_contrasts(ROOT, FILES, show=True, write=False)
R.report_interaction(ROOT, FILES, show=True)


tab:ex1:gaps   Contrasts, trial level (base minus manipulated)
----------------  ----------------  --------------------  --------------------  ------------------
Family            Level             Gemini                GPT                   Qwen              
----------------  ----------------  --------------------  --------------------  ------------------
Width removal     names true        +27.8 [+22.7, +33.2]  +24.0 [+18.5, +29.6]  -1.4 [-8.3, +5.6] 
Width removal     names anonymised  +24.7 [+19.8, +29.9]  +21.9 [+16.8, +27.2]  +4.2 [-2.9, +11.2]
Width removal     names swapped     +24.0 [+19.3, +29.3]  +19.9 [+14.7, +25.3]  +0.3 [-6.7, +7.4] 
Identity removed  width present     +0.0 [-1.3, +1.3]     -1.0 [-3.7, +1.5]     -1.4 [-8.3, +5.6] 
Identity removed  width absent      -3.1 [-10.3, +4.1]    -3.1 [-10.2, +4.0]    +4.2 [-2.9, +11.2]
Identity swapped  width present     +0.0 [-1.3, +1.3]     +0.1 [-2.8, +3.0]     +0.0 [-7.0, +7.0] 
Identity swapped  width absent      -3.7 [-10

In [12]:
# Scratch. Does each width-absent cell actually sit ON the width-blind line, or
# merely near it? The reference line is the strongest device in the chapter and
# the claim should be tested rather than eyeballed.
wb = REF["casta"]["width_blind"]
print("width-blind line %.1f\n" % wb)
for m in MODELS:
    for cond in ("nowidth", "nowidth-anon", "nowidth-swap"):
        k, n = R.legality(rows_for(m, cond))
        lo, hi = R.wilson(k, n)
        print("  %-7s %-14s %5.1f [%.1f, %.1f]  %s the line"
              % (R.MODEL_LABEL[m], cond, R.pct(k, n), lo, hi,
                 "INCLUDES" if lo <= wb <= hi else "excludes"))

width-blind line 74.9

  Gemini  nowidth         72.2 [66.8, 77.1]  INCLUDES the line
  Gemini  nowidth-anon    75.3 [70.1, 80.0]  INCLUDES the line
  Gemini  nowidth-swap    76.0 [70.7, 80.5]  INCLUDES the line
  GPT     nowidth         73.6 [68.1, 78.4]  INCLUDES the line
  GPT     nowidth-anon    76.7 [71.4, 81.2]  INCLUDES the line
  GPT     nowidth-swap    77.6 [72.4, 82.1]  INCLUDES the line
  Qwen    nowidth         76.7 [71.5, 81.2]  INCLUDES the line
  Qwen    nowidth-anon    72.6 [67.1, 77.4]  INCLUDES the line
  Qwen    nowidth-swap    75.0 [69.7, 79.6]  INCLUDES the line


---

# Q3. Does the model register the loss?

The spine of the chapter. Three readings of one question, and the data answers
all three the same way.

In [ ]:
R.table_signatures(ROOT, FILES, show=True, write=False)
R.prose_reasons(ROOT, FILES, show=True)

In [ ]:
# Scratch. The reasons bound is pooled across models in the script. Per model
# the denominators are very different, so the bounds are too. Worth knowing
# before writing a single number into the chapter.
print("Per-model bound on reasons admitting the missing width\n")
for m in MODELS:
    adm = ex = 0
    for cond in ("nowidth", "nowidth-anon"):
        for r in rows_for(m, cond):
            if r.get("violation_cause") != "grasp":
                continue
            ex += 1
            adm += bool(R.MISSING_INFO.search(r.get("model_reason") or ""))
    bound = 100 * 3 / ex if (adm == 0 and ex) else float("nan")
    print("  %-7s %d admit of %3d examined, one-sided 95%% upper bound %.1f%%"
          % (R.MODEL_LABEL[m], adm, ex, bound))
print("\nQwen's denominator is much smaller, so its bound is much weaker.")
print("Do not let the pooled 0.5% stand in for all three models.")

In [ ]:
# Scratch. A sample of the reason strings, so the claim "not one admits the gap"
# is inspectable rather than taken on trust from a regex.
seen = 0
for r in rows_for("gpt", "nowidth"):
    if r.get("violation_cause") != "grasp":
        continue
    print("-", (r.get("model_reason") or "")[:140])
    seen += 1
    if seen >= 8:
        break

---

# Q4. Can the object's name supply what the number supplied?

In [ ]:
R.table_swap(ROOT, FILES, show=True, write=False)

In [ ]:
# Scratch. Per object rather than pooled by direction. Small n per object, so
# this is for inspection, not for the chapter.
true_object = {}
for p in probes:
    pv = p["provenance"]
    for t in p["state"]["tasks"]:
        true_object[(pv["source"], pv["seq"], pv["round"], t["id"])] = t["object"]

def by_object(rs):
    tot, fr = collections.Counter(), collections.Counter()
    for r in rs:
        d = r.get("decision") or {}
        arm, tid = d.get("arm"), d.get("task_id")
        if not arm or tid is None:
            continue
        pv = r["provenance"]
        o = true_object.get((pv["source"], pv["seq"], pv["round"], tid))
        if o is None:
            continue
        tot[o] += 1
        fr[o] += arm.startswith("franka")
    return fr, tot

model = "gpt"
fb, tb = by_object(rows_for(model, "nowidth"))
fs, ts = by_object(rows_for(model, "nowidth-swap"))
# The three swapped pairs, read from the experiment module so the notebook
# cannot drift from what was actually rendered into the prompts.
sys.path.insert(0, os.path.join(ROOT, "experiments", "ex1"))
try:
    import mislabel
    PARTNER = dict(mislabel.SWAP_MAP) if hasattr(mislabel, "SWAP_MAP") else {}
except Exception as exc:
    print("could not import mislabel (%s), falling back to the pair list" % exc)
    PARTNER = {}
if not PARTNER:
    PARTNER = {"ycb_large_clamp": "ycb_power_drill",
               "ycb_power_drill": "ycb_large_clamp",
               "ycb_mustard": "ycb_soup_can",
               "ycb_soup_can": "ycb_mustard",
               "ycb_meat_can": "ycb_gelatin_box",
               "ycb_gelatin_box": "ycb_meat_can"}

headers = ["Object", "True width", "Presented as", "Base Franka %",
           "Swap Franka %", "n base/swap"]
rows = []
for obj, w in R.SWAP_PAIRS:
    rows.append([obj.replace("ycb_", ""), "%.3f" % w,
                 PARTNER.get(obj, "?").replace("ycb_", ""),
                 "%.1f" % R.pct(fb[obj], tb[obj]),
                 "%.1f" % R.pct(fs[obj], ts[obj]),
                 "%d/%d" % (tb[obj], ts[obj])])
show(headers, rows, "%s, width absent, per swapped object" % R.MODEL_LABEL[model])

---

# Q5. Can the rule text supply it?

In [ ]:
R.table_composition(ROOT, FILES, show=True, write=False)

In [ ]:
# Scratch. The composition table counts violations across ALL states. Split by
# state type, because a violation on a zero-legal state is a failure to refuse
# and a violation on a picking state is a wrong choice. They are different
# failures and pooling them hides the difference.
for m in MODELS:
    for cond in ("full", "nowidth", "norules"):
        onzero, onpick = collections.Counter(), collections.Counter()
        for r in rows_for(m, cond):
            vc = r.get("violation_cause")
            if not vc:
                continue
            (onzero if r.get("zero_legal") else onpick)[vc] += 1
        print("%-7s %-10s picking %-46s zero-legal %s"
              % (R.MODEL_LABEL[m], cond, dict(onpick) or "none",
                 dict(onzero) or "none"))
    print()

---

# Q6. Which parts of the effect are object-set dependent?

In [ ]:
R.table_castb(ROOT, FILES, REF, show=True, write=False)

In [ ]:
# Scratch. The independent single-repeat run against repeat 1 of the
# three-repeat run: same states, same prompt version, different session. How
# often does GPT give a different answer to the same question?
for cond in R.CASTB_CONDITIONS:
    a = {R.scene_key(r): r for r in R.load(ROOT, FILES[("castb_r1", "gpt", cond)])}
    b = {R.scene_key(r): r for r in R.load(ROOT, FILES[("castb", "gpt", cond)])
         if r.get("repeat", 1) == 1}
    diff = sum(1 for k in a
               if (a[k].get("result"), (a[k].get("decision") or {}).get("arm"))
               != (b[k].get("result"), (b[k].get("decision") or {}).get("arm")))
    print("  %-14s %3d of %3d states answered differently (%.0f%%)"
          % (cond, diff, len(a), 100 * diff / len(a)))

---

# Robustness checks, and writing everything out

In [ ]:
R.prose_image(ROOT, FILES, IMAGE, show=True)
R.reconcile(ROOT, FILES, show=True)
R.summarise_checks()

In [ ]:
# Write the LaTeX and CSV. Everything above ran with write=False, so nothing has
# touched disk yet. Files land under the PACKAGE root, not next to this
# notebook, so \input{tables/ex1_design} keeps working from main.tex.
CONFIRM = False          # set True to write

if CONFIRM:
    R.WRITTEN.clear()
    R.table_constraints(ROOT, FILES, False, True)
    R.table_design(ROOT, FILES, REF, False, True)
    R.table_contrasts(ROOT, FILES, False, True)
    R.table_signatures(ROOT, FILES, False, True)
    R.table_swap(ROOT, FILES, False, True)
    R.table_composition(ROOT, FILES, False, True)
    R.table_castb(ROOT, FILES, REF, False, True)
    R.figure_convergence(ROOT, FILES, REF, False, True, True)
    for p in R.WRITTEN:
        print("wrote", p)
    print("\nIn main.tex:")
    for p in R.WRITTEN:
        if p.endswith(".tex"):
            print("    \\input{%s}" % p[:-len(".tex")])
else:
    print("CONFIRM is False, nothing written. "
          "Set it True and re-run this cell when you are ready.")

---

## Open decisions

1. **Q1.** Replace `tab:ex1:constraints` with the competence profile, or keep
   both? If replaced, the label becomes `tab:ex1:baseline`.
2. **Q3.** Does `fig:ex1:convergence` lead and the table support, or the other
   way round?
3. **Q5.** Does `tab:ex1:composition` earn its place once its Q2 job is already
   done by the negative-control column in `tab:ex1:design`?
4. **Legal-Arm Control** is one repeat while everything else is three. Footnote
   it, or spend 324 calls on Qwen, whose cell now carries real weight.